# Fine-Tuning with Axolotl (LoRA & QLoRA)

**Axolotl** is a config-driven fine-tuning framework: instead of writing
training code you write a **YAML config** and Axolotl handles everything —
data loading, tokenisation, model loading, LoRA/QLoRA, DeepSpeed, etc.

This approach excels at reproducibility and rapid experimentation but has
different failure modes than writing code by hand — mostly around YAML
correctness and dataset format matching.

**This notebook covers:**
- Anatomy of an Axolotl YAML config
- LoRA vs QLoRA vs 8-bit LoRA via config keys
- Validating with `axolotl check-data` before training
- Running training and reading MLflow results

> **GPU required (Ampere+ recommended).** Flash-attention needs SM ≥ 80.
> On older GPUs, see the hardware gotchas section.

## Environment Setup

Run once in a terminal before opening this notebook:

```bash
cd projects/axolotl_finetuning
uv sync --no-install-workspace   # installs torch==2.6.0, axolotl, flash-attn, etc.
uv run ipython kernel install --user --env VIRTUAL_ENV ../../.venv --name=axolotl_finetuning
```

Then start the MLflow server (optional, but recommended for tracking):

```bash
bash .devcontainer/start_mlflow.sh
```

Select the **`axolotl_finetuning`** kernel in JupyterLab.

In [ ]:
import sys, subprocess, pathlib, yaml, torch

print(f'Python {sys.version}')
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    cc = f'{props.major}.{props.minor}'
    print(f'GPU: {props.name}  SM{cc}  VRAM: {props.total_memory / 1e9:.1f} GB')
    if props.major < 8:
        print('WARNING: SM < 8.0 — flash_attention will not work. Set flash_attention: false in config.')

import axolotl
print(f'Axolotl: {axolotl.__version__}')

result = subprocess.run(['axolotl', '--help'], capture_output=True, text=True)
print(f'Axolotl CLI available: {result.returncode == 0}')

In [ ]:
import sys, pathlib, os
sys.path.insert(0, str(pathlib.Path('../../../').resolve()))
from dotenv_config import dot_env_settings

if dot_env_settings.HF_TOKEN:
    os.environ['HF_TOKEN'] = dot_env_settings.HF_TOKEN
    print('HF_TOKEN loaded.')
else:
    print('WARNING: No HF_TOKEN. Private/gated models will fail.')

## Anatomy of an Axolotl YAML Config

The config has six logical sections:

| Section | Key fields |
|---------|------------|
| **Model** | `base_model`, `model_type`, `load_in_4bit` / `load_in_8bit` |
| **Dataset** | `datasets[].path`, `datasets[].type`, `datasets[].split` |
| **Adapter** | `adapter`, `lora_r`, `lora_alpha`, `lora_target_modules` |
| **Training** | `num_epochs`, `micro_batch_size`, `optimizer`, `learning_rate` |
| **Hardware** | `bf16`, `flash_attention`, `gradient_checkpointing` |
| **Output** | `output_dir`, `mlflow_tracking_uri`, `save_steps` |

> **Gotcha #1 — YAML key typos are silently ignored.**
> Axolotl validates config with Pydantic. Unknown keys are **silently dropped**
> (not errored) unless you pass `--strict`. A typo like `lora_alhpa: 32`
> will be ignored and the default value used instead — wasting a training run.
> Always run `axolotl check-data` first (see below).

> **Gotcha #2 — `datasets[].type` must match the actual data format.**
> `type: alpaca` expects `{instruction, input, output}` fields.
> `type: sharegpt` expects `{conversations: [{from, value}, ...]}` fields.
> Mixing these causes cryptic tokenisation errors deep in the training loop.

## LoRA vs QLoRA vs 8-bit LoRA in Axolotl

All three are set via two YAML keys — `load_in_4bit` / `load_in_8bit` and
`adapter: lora`:

```yaml
# Standard LoRA (float16 base model)
load_in_8bit: false
load_in_4bit: false
adapter: lora

# 8-bit LoRA (LLM.int8 base + LoRA adapters)
load_in_8bit: true
load_in_4bit: false
adapter: lora

# QLoRA (4-bit NF4 base + LoRA adapters) ← used in this notebook
load_in_8bit: false
load_in_4bit: true
adapter: lora
```

| | LoRA | 8-bit LoRA | QLoRA (4-bit) |
|-|------|------------|---------------|
| Base VRAM (1.1B) | ~2.2 GB | ~1.5 GB | ~0.7 GB |
| Quality | Best | Good | ~2–5% lower |
| Speed | Fastest | Moderate | Slightly slower |

The notebook below uses **QLoRA** (`load_in_4bit: true`).
Change `load_in_4bit: false` in the config dict to switch to standard LoRA.

## Dataset Format Reference

**Alpaca format** (`type: alpaca`):
```json
{"instruction": "...", "input": "...", "output": "..."}
```

**ShareGPT format** (`type: sharegpt`):
```json
{"conversations": [{"from": "human", "value": "..."}, {"from": "gpt", "value": "..."}]}
```

We use `mhenrichsen/alpaca_data_cleaned` from the HuggingFace Hub with
`type: alpaca`.

> **Gotcha #3 — `output_dir` must exist before training.**
> Axolotl does not create deeply nested directories automatically.
> The next cell creates both the prepared-dataset cache dir and the output dir.

In [ ]:
import pathlib

for d in ['/tmp/axolotl_prepared', '/tmp/axolotl_output']:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
    print(f'Ready: {d}')

## Writing the Config

Building the config as a Python dict makes it easy to version-control and
diff. PyYAML writes it to disk for the Axolotl CLI.

In [ ]:
config = {
    'base_model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'model_type': 'LlamaForCausalLM',
    'tokenizer_type': 'LlamaTokenizer',

    # Quantization — change load_in_4bit to False for standard LoRA
    'load_in_8bit': False,
    'load_in_4bit': True,
    'strict': False,

    'datasets': [{
        'path': 'mhenrichsen/alpaca_data_cleaned',
        'type': 'alpaca',    # MUST match actual data fields
        'ds_type': 'parquet',
        'split': 'train',
        'shards': 1,
    }],
    'dataset_prepared_path': '/tmp/axolotl_prepared',
    'val_set_size': 0.05,
    'output_dir': '/tmp/axolotl_output',

    'sequence_len': 512,
    'sample_packing': False,

    # Adapter
    'adapter': 'lora',
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'lora_target_modules': ['q_proj', 'v_proj'],

    # Training
    'gradient_accumulation_steps': 4,
    'micro_batch_size': 2,
    'num_epochs': 1,
    'optimizer': 'adamw_bnb_8bit',
    'lr_scheduler': 'cosine',
    'learning_rate': 2e-4,
    'train_on_inputs': False,
    'bf16': 'auto',
    'fp16': None,
    'gradient_checkpointing': True,
    'logging_steps': 10,

    # Hardware (set flash_attention: False on GPUs older than Ampere)
    'flash_attention': True,
    'xformers_attention': None,

    # Checkpointing
    'warmup_steps': 10,
    'eval_steps': 50,
    'save_steps': 100,
    'save_total_limit': 2,

    # Tracking
    'mlflow_tracking_uri': 'http://127.0.0.1:1235',
    'mlflow_experiment_name': 'axolotl-tinyllama-qlora',
}

CONFIG_PATH = pathlib.Path('axolotl_config.yaml')
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f'Config written to {CONFIG_PATH.resolve()}')
print(CONFIG_PATH.read_text())

## Always Validate Before Training: `axolotl check-data`

> **Gotcha #4 — Never skip `axolotl check-data`.**
> This command validates the YAML structure, downloads a sample of your
> dataset, tokenises it, and shows you the formatted prompts. It catches
> `type` mismatches and malformed data **before** you waste GPU time.
> Takes ~30 seconds. Skipping it is the #1 cause of wasted training runs.

In [ ]:
result = subprocess.run(
    ['axolotl', 'check-data', str(CONFIG_PATH)],
    capture_output=True, text=True,
)
# Show last 3000 chars to avoid flooding the cell
output = result.stdout + result.stderr
print(output[-3000:] if len(output) > 3000 else output)

if result.returncode != 0:
    raise RuntimeError(f'axolotl check-data failed (exit {result.returncode})')
print('Validation passed!')

## Hardware Gotchas

> **Gotcha #5 — `flash_attention: true` requires Ampere+ GPU (SM ≥ 8.0).**
> Ampere = A100, A10, RTX 30xx, RTX 40xx. Older cards (V100 = SM 7.0,
> T4 = SM 7.5) are not supported. If you see a CUDA error mentioning
> `flash_attn_varlen_func`, set `flash_attention: false` and optionally
> `xformers_attention: true` as a faster-than-standard fallback.

> **Gotcha #6 — Don't use DeepSpeed on a single GPU.**
> DeepSpeed ZeRO shards model state across GPUs/nodes. On one GPU it adds
> communication overhead with no benefit. Leave the `deepspeed:` key absent
> from your config for single-GPU runs.

## Running Training

Axolotl training is invoked via `axolotl train <config>`. The cell below
runs it as a subprocess and streams output line-by-line.

For long runs (>15 min) consider running from a terminal instead so the
notebook kernel timeout doesn't interrupt training:
```bash
axolotl train notebooks/axolotl_config.yaml
```

In [ ]:
process = subprocess.Popen(
    ['axolotl', 'train', str(CONFIG_PATH)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'Training failed (exit {process.returncode})')
print('Training complete!')

## MLflow Integration

When the MLflow server is running (`bash .devcontainer/start_mlflow.sh`),
Axolotl logs loss, learning rate, and config automatically.

Navigate to **http://localhost:1235** to view the dashboard.

In [ ]:
# Optional: query MLflow programmatically
try:
    import mlflow
    mlflow.set_tracking_uri('http://127.0.0.1:1235')
    client = mlflow.tracking.MlflowClient()
    exp = client.get_experiment_by_name('axolotl-tinyllama-qlora')
    if exp:
        runs = client.search_runs(exp.experiment_id, order_by=['start_time DESC'], max_results=3)
        for r in runs:
            print(f'Run {r.info.run_id[:8]}  loss={r.data.metrics.get("train/loss", "n/a")}')
    else:
        print('Experiment not found — is MLflow running?')
except Exception as e:
    print(f'MLflow unavailable: {e}')

## Understanding the Output Directory

After training, `/tmp/axolotl_output` contains:

```
axolotl_output/
├── adapter_model.safetensors   # LoRA weights
├── adapter_config.json          # PEFT adapter configuration
├── tokenizer.json               # tokenizer copy
├── tokenizer_config.json
├── checkpoint-100/              # intermediate checkpoint (save_steps=100)
└── last_run_prepared/           # preprocessed dataset cache
```

To merge the adapter into the base model for serving:
```bash
axolotl merge-lora axolotl_config.yaml --lora-model-dir /tmp/axolotl_output
```

In [ ]:
output_dir = pathlib.Path('/tmp/axolotl_output')
if output_dir.exists():
    for f in sorted(output_dir.rglob('*')):
        rel = f.relative_to(output_dir)
        if f.is_file():
            print(f'  {rel}  ({f.stat().st_size:,} bytes)')
        else:
            print(f'  {rel}/')
else:
    print('Output directory not found — did training complete?')

## Summary

### Gotcha recap

| # | Gotcha | Fix |
|---|--------|-----|
| 1 | YAML key typos silently ignored | Run with `--strict`; always use `check-data` |
| 2 | `type:` mismatch with data format | Check field names: alpaca vs sharegpt |
| 3 | `output_dir` must exist | Create with `pathlib.Path(...).mkdir(parents=True)` |
| 4 | Skipping `check-data` | Always run it before `axolotl train` |
| 5 | `flash_attention` on old GPUs | Use `xformers_attention: true` on SM < 8.0 |
| 6 | DeepSpeed on single GPU | Omit the `deepspeed:` key entirely |

### LoRA / QLoRA / 8-bit LoRA quick reference

```yaml
# Standard LoRA
load_in_4bit: false
load_in_8bit: false

# QLoRA (this notebook)
load_in_4bit: true
load_in_8bit: false

# 8-bit LoRA
load_in_4bit: false
load_in_8bit: true
```

**Next steps:** swap `type: alpaca` for `type: sharegpt` with a conversation
dataset, add `deepspeed:` config for multi-GPU, try `axolotl merge-lora` to
produce a standalone model.